# threshold-logic-unit

A tiny, faithful replication of the paper that **invented the artificial neuron** — run the cells top to bottom and watch each idea work.

> Warren S. McCulloch & Walter Pitts (1943). *A Logical Calculus of the Ideas Immanent in Nervous Activity.* The Bulletin of Mathematical Biophysics 5(4):115–133. [doi:10.1007/BF02478259](https://doi.org/10.1007/BF02478259)

**The big idea:** a neuron is *all-or-none* — it either fires or it doesn't. So “this neuron fired” is just a **true/false** statement, and a network of neurons is a circuit that computes **logic**.

## The neuron

Their neuron has **no weights** and does **not learn**. It just **counts** its active excitatory inputs and fires when the count reaches a fixed threshold `θ` — *unless* an inhibitory input is active, which **absolutely vetoes** firing.

```
fire = (count of active excitatory inputs ≥ θ)  AND  (no inhibitor active)
```

In [1]:
def mp(exc, inh, theta):
    """Fire (1) iff threshold met AND no inhibitor active. No weights, no learning."""
    return int(not any(inh) and sum(exc) >= theta)

# a neuron that needs BOTH inputs (threshold 2): on for (1,1), off for (1,0)
mp([1, 1], [], 2), mp([1, 0], [], 2)

(1, 0)

## Part 1 — one neuron *is* a logic gate

Pick the threshold and the same neuron becomes a different gate.

In [2]:
AND = lambda a, b: mp([a, b], [], 2)   # needs BOTH inputs  -> threshold 2
OR  = lambda a, b: mp([a, b], [], 1)   # needs EITHER input -> threshold 1
NOT = lambda a:    mp([], [a], 0)      # fires by default; its input inhibits it

In [3]:
bits = [(0, 0), (0, 1), (1, 0), (1, 1)]

for name, fn in [("AND", AND), ("OR", OR)]:
    print(name)
    for a, b in bits:
        print(f"  {a} {b} -> {fn(a, b)}")

print("NOT")
for a in (0, 1):
    print(f"  {a} -> {NOT(a)}")

AND
  0 0 -> 0
  0 1 -> 0
  1 0 -> 0
  1 1 -> 1
OR
  0 0 -> 0
  0 1 -> 1
  1 0 -> 1
  1 1 -> 1
NOT
  0 -> 1
  1 -> 0


## Part 2 — a network computes *anything* (XOR)

One neuron **cannot** do XOR (it isn't linearly separable). But wire a few together and you can build any logic at all:

```
a XOR b = (a OR b) AND NOT(a AND b)
```

In [4]:
def XOR(a, b):
    return AND(OR(a, b), NOT(AND(a, b)))   # three gates, one little network

print("XOR")
for a, b in bits:
    print(f"  {a} {b} -> {XOR(a, b)}")

XOR
  0 0 -> 0
  0 1 -> 1
  1 0 -> 1
  1 1 -> 0


## Part 3 — a loop gives memory

Feed a neuron's own output back into itself. Once **set**, it keeps re-firing (**reverberates**) and so *remembers* — until **reset** inhibits it. The authors call this firing “a memory — or an idea.”

In [5]:
def memory(events):
    state, timeline = 0, []
    for set_, reset_ in events:                  # each tick: (set?, reset?)
        state = mp([set_, state], [reset_], 1)   # fire if set OR still-on, unless reset
        timeline.append(state)
    return timeline

events = [(0, 0), (1, 0), (0, 0), (0, 0), (0, 1), (0, 0)]  # set@t1, reset@t4
print("set/reset:", events)
print("state:    ", memory(events))

set/reset: [(0, 0), (1, 0), (0, 0), (0, 0), (0, 1), (0, 0)]
state:     [0, 1, 1, 1, 0, 0]


## The math must match the paper

A few assertions so the notebook is self-checking — if any output is wrong, the cell errors.

In [6]:
assert [AND(a, b) for a, b in bits] == [0, 0, 0, 1]
assert [OR(a, b) for a, b in bits]  == [0, 1, 1, 1]
assert [NOT(a) for a in (0, 1)]     == [1, 0]
assert [XOR(a, b) for a, b in bits] == [0, 1, 1, 0]
assert memory(events) == [0, 1, 1, 1, 0, 0]
print("All checks pass -- one neuron, four ideas, true to 1943.")

All checks pass -- one neuron, four ideas, true to 1943.


---

A network of these neurons is exactly a **finite-state machine** (logic + memory); add an external tape and it's **Turing complete** — the bridge from brains to computers, in 1943. The next step in the story is [Rosenblatt's perceptron (1958)](https://doi.org/10.1037/h0042519): add **weights + a learning rule** and the neuron can *learn from data*.

*Educational reconstruction by [Average Joes Lab](https://averagejoeslab.com). All credit for the ideas to McCulloch & Pitts (1943).*